# A sample to label by hand

A hundred replies from each model, drawn to be labelled by a person, so that
the classifier can be measured against something other than itself.

Every number in the results passes through the classifier, so its accuracy is
the accuracy of the study. The only way to establish that is to label a sample
by hand and compare. This notebook draws the sample and, once the labelling is
done, reads it back and reports the agreement.

**The same hundred prompts for every model.** A prompt is only eligible if
every model answered it, so the files line up row for row. That is more work to
select and worth it twice over: the models become comparable on identical items,
and a disagreement between the classifier and you can be read across all six at
once rather than one file at a time.

Replicate 1 only, and nothing blocked, errored or empty. A reply that never
arrived cannot be labelled, and including it would put the classifier's handling
of an absent reply into a figure meant to measure its reading of a present one.

A CSV per model with the label columns blank, and one more holding the replies
a provider withheld. Those are already labelled and are not for reading: nobody
decided them, so a label from you and a label from the classifier would agree by
construction and measure nothing.

In [1]:
# Import the libraries
import sys
from pathlib import Path

import pandas as pd

In [2]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

/Users/rinlobachevskii/Desktop/Git/Thesis


In [3]:
# Import the pipeline
%load_ext autoreload
%autoreload 2

import settings
import utils

ANNOTATION_DIR = settings.RESULTS_DIR / 'annotation'
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 70)
print('Ready')

Ready


## Read what was collected

Replicate 1 of every model, with the replies that never arrived left out.

In [4]:
collected = []
for path in sorted(settings.ADAPTATION_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.ADAPTATION_DIR}')

replies = pd.concat(collected, ignore_index=True)
replies['response'] = replies['response'].astype(str)
replies['blocked'] = replies['blocked'].astype(str)

# What the provider withheld, at whatever replicate it happened. A prompt whose
# third draw was blocked still has a first and a second, so only the blocked
# draw is treated this way and the others stay eligible like any other.
withheld = replies[replies['blocked'].str.strip() != ''].copy()

first = replies[replies['replicate'].astype(str) == '1']
answered = first[(first['response'].str.strip() != '')
                 & (first['blocked'].str.strip() == '')
                 & (first['error'].astype(str).str.strip() == '')]

print(f'{len(replies):,} replies, {len(withheld)} withheld by a provider')
display(withheld.groupby(['model', 'blocked']).size().rename('replies').to_frame())
print(f'\n{len(answered):,} answered at replicate 1, of {len(first):,}')

46,800 replies, 160 withheld by a provider


replies
model                     blocked                    
claude-haiku-4-5-20251001 CONTENT_FILTER            1
gemini-3.5-flash-lite     PROHIBITED_CONTENT      158
                          RECITATION                1


15,547 answered at replicate 1, of 15,600


## Only prompts every model answered

The sample is the same for all of them, so a prompt one model failed to answer
is dropped for all. Gemini's blocked prompts are most of what goes here, and
losing them from the labelled sample does not lose them from the results: they
are reported as their own outcome elsewhere.

In [5]:
answered_by = answered.groupby('prompt_id')['model'].nunique()
models = answered.groupby('model').ngroups
shared = set(answered_by[answered_by == models].index)

print(f'{len(shared):,} prompts answered at replicate 1 by all {models} models, '
      f'of {answered["prompt_id"].nunique():,}')
print('The rest had at least one model that did not answer, and are left out of')
print('the shared draw so that the files line up. They are not lost: a prompt')
print('withheld at one replicate is in the section above at that replicate.')

2,547 prompts answered at replicate 1 by all 6 models, of 2,600
The rest had at least one model that did not answer, and are left out of
the shared draw so that the files line up. They are not lost: a prompt
withheld at one replicate is in the section above at that replicate.


## Draw the hundred

Allocated across the four strata in proportion to the benchmark, then spread
across domain and condition inside each one, so the sample looks like the
benchmark rather than like whichever rows sorted first.

The earlier draw gave every domain by band cell a floor of one row and then cut
the result back to a hundred. Because groups come back in sorted key order, the
cut fell on whichever stratum sorted last, and that was Rights: the largest of
the four at 75 of 200 scenarios, and the one the over-refusal claim rests on. It
is also where six of the twelve properties actually occur, so its absence left
those fields with too few positives to produce a usable kappa. The draw below
allocates first and truncates never, and asserts both facts so the fault cannot
return unnoticed.


In [6]:
HOW_MANY = 100

prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)
frame = (prompts[prompts['prompt_id'].isin(shared)]
         .merge(benchmark[['scenario_id', 'domain', 'scenario_type', 'category']],
                on='scenario_id'))

# Allocate the hundred across the strata in proportion to the benchmark, by
# largest remainder so the parts sum to exactly HOW_MANY without a final cut.
sizes = frame['scenario_type'].value_counts()
exact = sizes / sizes.sum() * HOW_MANY
quota = exact.astype(int)
for name in (exact - quota).sort_values(ascending=False).index[:HOW_MANY - quota.sum()]:
    quota[name] += 1

# Inside a stratum, spread before filling, on two keys. Rows are ranked within
# their domain and within their condition, and the lowest ranks are taken first,
# so every domain is drawn once before any domain is drawn twice and conditions
# vary within that. Domain is the primary key because a stratum has fewer slots
# than it has domain by condition cells, so one of the two has to give, and a
# missing domain costs more than a missing condition: conditions are covered
# across the whole hundred, domains would not be.
parts = []
for scenario_type, rows in frame.groupby('scenario_type'):
    shuffled = rows.sample(frac=1, random_state=settings.SEED).copy()
    shuffled['domain_rank'] = shuffled.groupby('domain').cumcount()
    shuffled['condition_rank'] = shuffled.groupby('condition').cumcount()
    parts.append(shuffled.sort_values(['domain_rank', 'condition_rank'])
                 .head(quota[scenario_type]))
chosen = pd.concat(parts, ignore_index=True)

# The three things that went wrong before, checked rather than trusted.
assert len(chosen) == HOW_MANY, f'{len(chosen)} drawn, expected {HOW_MANY}'
assert set(chosen['scenario_type']) == set(frame['scenario_type']), \
    f'missing strata: {set(frame["scenario_type"]) - set(chosen["scenario_type"])}'
assert set(chosen['domain']) == set(frame['domain']), \
    f'missing domains: {set(frame["domain"]) - set(chosen["domain"])}'

print(f'{len(chosen)} prompts, {chosen["scenario_id"].nunique()} distinct scenarios, '
      f'{chosen["condition"].nunique()} of {frame["condition"].nunique()} conditions\n')
display(pd.crosstab(chosen['domain'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
print()
display(chosen['condition'].value_counts().sort_index()
        .rename('prompts').to_frame().T)


100 prompts, 83 distinct scenarios, 13 of 13 conditions



scenario_type,Age Restricted,Benign,Harmful,Rights,all
domain,,,,,
Abuse & Hate,0,3,3,4,10
Body Image,3,2,2,3,10
Bullying,0,3,3,4,10
Dangerous Challenges,3,3,2,4,12
Eating Disorders,0,3,2,4,9
Emotional Dependency,0,2,3,4,9
Harmful Substances,2,3,3,3,11
Self-Harm & Suicide,0,2,2,4,8
Sexual Content,2,2,2,4,10


condition,age07,age09,age11,age13,age15,age17,age18,age21,neutral,people_adult,people_minor,routine_adult,routine_minor
prompts,14,7,7,8,6,9,4,13,6,10,6,6,4


## Write one file per model

Same rows, same order, one model each. The label columns are empty and are for
you: fill in `answer` and the five properties as you read, leaving anything you
are unsure of blank rather than guessing, since a doubtful label is worse than a
missing one when the point is to measure a classifier against it.

`expected_answer` is included because it is part of the benchmark, not a hint.
Label what the reply did, not what it should have done.

In [7]:
LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]
# The same order as everything else the pipeline writes: model, prompt and
# replicate identify a row, then who judged it, then what was said and what was
# decided. A labelled sheet is a judgement file with the text kept beside it, so
# the two line up column for column across the part they share.
#
# Nothing derived or derivable is stored. The scenario type, domain, category
# and condition are in the prompt identifier and in benchmark.csv; the expected
# answer is in prompts.csv. Keeping them out is not only tidiness: an expected
# answer beside the label would nudge it, and a stored one would go stale the
# moment a stratum is revised. What a reply did is an observation; whether it
# should have is a comparison, and the comparison is made where the two meet.
BEFORE = ['model', 'prompt_id', 'replicate', 'judge', 'prompt', 'response']
SHEET_COLUMNS = BEFORE

facts = prompts[['prompt_id', 'prompt']]
WHO = 'human'          # whose labels these are, beside the classifier's name
drawn = set(chosen['prompt_id'])


# Define function to lay out one model's rows in the order they will be read.
# The reason a provider gave is carried through, because a prompt can be
# withheld at one replicate and answered at another, and the row is about the
# replicate rather than the prompt.
def sheet_for(rows):
    sheet = rows.merge(facts, on='prompt_id')
    sheet['note'] = sheet['blocked'].astype(str)
    sheet['judge'] = WHO
    for column in LABEL_COLUMNS:
        sheet[column] = ''
    return (sheet[BEFORE + LABEL_COLUMNS + ['note']]
            .sort_values(['prompt_id', 'replicate']))


# The withheld replies are recorded once, together, and are not for labelling.
# Nobody decided them: the provider did, so a label from you and a label from
# the classifier would agree by construction and tell you nothing. They are here
# so the set is visible and countable, not so it can be read.
withheld_sheet = sheet_for(withheld)
if len(withheld_sheet):
    withheld_sheet['answer'] = settings.BLOCKED
    path = ANNOTATION_DIR / 'blocked.csv'
    withheld_sheet.to_csv(path, index=False)
    print(f'  {len(withheld_sheet):>4} withheld, recorded in {path.name}, '
          f'not for labelling')
    print(f'       {withheld_sheet.groupby("model").size().to_dict()}\n')

for model, group in replies.groupby('model'):

    # then the shared draw, at replicate 1, for you to read and label
    to_label = sheet_for(group[(group['prompt_id'].isin(drawn))
                               & (group['replicate'].astype(str) == '1')
                               & (group['blocked'].str.strip() == '')])
    to_label['note'] = ''

    path = ANNOTATION_DIR / f'{utils.model_slug(model)}.csv'
    to_label.to_csv(path, index=False)
    print(f'  {model:<28} {len(to_label):>4} to label  {path.name}')

print(f'\nWritten to {ANNOTATION_DIR}')
print(f'Columns to fill: {", ".join(LABEL_COLUMNS)}')
print(f'The {settings.BLOCKED} rows are filled in already. Leave them.')

   160 withheld, recorded in blocked.csv, not for labelling
       {'claude-haiku-4-5-20251001': 1, 'gemini-3.5-flash-lite': 159}

  claude-haiku-4-5-20251001     100 to label  claude-haiku-4-5-20251001.csv
  deepseek-v4-flash             100 to label  deepseek-v4-flash.csv
  gemini-3.5-flash-lite         100 to label  gemini-3.5-flash-lite.csv
  gemma4:31b-cloud              100 to label  gemma4-31b-cloud.csv
  gpt-5.6-luna                  100 to label  gpt-5.6-luna.csv
  mistral-small-2603            100 to label  mistral-small-2603.csv

Written to /Users/rinlobachevskii/Desktop/Git/Thesis/results/annotation
Columns to fill: answer, delivery_response, alternative_response, risk_statement, legal_statement, eligibility_statement, social_signpost, expert_signpost, service_signpost, system_identity, boundary_identity, limitation_identity, companion_identity
The Blocked rows are filled in already. Leave them.


## Agreement, over every hand-labelled row

Six models, a hundred replies each. The classifier is run over the same six
hundred and compared field by field.

Nothing here writes to the folder the sheets live in. Results go to
`results/annotation judge/`, so a rerun cannot overwrite a label.

In [ ]:
MANUAL_DIR = settings.RESULTS_DIR / 'annotation manual'
JUDGE_DIR = settings.RESULTS_DIR / 'annotation judge'
JUDGE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]

# The sheets carry three columns that are notes to the annotator rather than
# labels: whether a row was marked unsure, a free-text comment, and the flag the
# pre-annotator raised. They are read here so that unsure rows can be excluded
# from agreement, and they are never compared.
ASIDE = ['uncertain', 'comment', 'note']

# Loaded here as well as in the drawing half above, so the agreement cells can
# be run on their own without redrawing a sample.
prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)

sheets = []
for path in sorted(MANUAL_DIR.glob('*_human_labels.csv')):
    sheet = pd.read_csv(path, dtype=str, keep_default_na=False)
    sheet['replicate'] = pd.to_numeric(sheet['replicate'], errors='coerce')
    sheets.append(sheet)

if not sheets:
    raise SystemExit(f'No *_human_labels.csv in {MANUAL_DIR}')

manual = pd.concat(sheets, ignore_index=True)
manual = manual[manual['answer'].str.strip() != '']
manual['stratum'] = manual['prompt_id'].str.split('-').str[1].str[0].map(
    {'h': 'Harmful', 'a': 'Age Restricted', 'r': 'Rights', 'b': 'Benign'})

print(f'{len(manual)} labelled rows across {manual["model"].nunique()} models')
for model, group in manual.groupby('model'):
    unsure = int((group.get('uncertain', pd.Series('', index=group.index))
                  == 'unsure').sum()) if 'uncertain' in group else 0
    print(f'  {model:<28} {len(group):>4} rows, {unsure:>2} marked unsure')

### Score the same rows with the classifier

The classifier receives the bare scenario request, not the prompt with the age
opener, so it sees what it will see in the full pass. Verdicts are written to
`results/annotation judge/` and reused on a rerun, so a broken cell below costs
no calls.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

import evaluate

BACKEND = 'ollama'
WORKERS = 16
CACHE = JUDGE_DIR / 'verdicts.csv'

# The bare request, as the classifier gets it in the full pass. The prompt
# column carries the age opener and is deliberately not used: the classifier
# describes what a reply did, and the age is joined afterwards in analysis.
scenario = dict(zip(prompts['prompt_id'], prompts['scenario_id']))
request = dict(zip(benchmark['scenario_id'], benchmark['request']))

if CACHE.exists():
    judged = pd.read_csv(CACHE, dtype=str, keep_default_na=False)
    print(f'{len(judged)} verdicts read from {CACHE.name}, no calls made')
else:
    def score(row):
        try:
            verdict = evaluate.judge_reply(
                judge=settings.JUDGE['id'], reply=row.response, backend=BACKEND,
                request=request[scenario[row.prompt_id]])
        except Exception as problem:                       # noqa: BLE001
            return {'unreadable': str(problem)[:120]}
        return verdict

    started = time.perf_counter()
    rows = list(manual.itertuples())
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        verdicts = list(pool.map(score, rows))
    elapsed = time.perf_counter() - started

    judged = pd.DataFrame(verdicts).fillna('')
    for column in LABEL_COLUMNS:
        if column not in judged:
            judged[column] = ''
    judged.insert(0, 'replicate', manual['replicate'].to_numpy())
    judged.insert(0, 'prompt_id', manual['prompt_id'].to_numpy())
    judged.insert(0, 'model', manual['model'].to_numpy())
    judged.to_csv(CACHE, index=False)

    unread = int((judged.get('unreadable', pd.Series('', index=judged.index))
                  .astype(str).str.strip() != '').sum())
    print(f'{len(judged)} replies scored in {elapsed / 60:.1f} minutes '
          f'at {WORKERS} workers, {unread} unreadable')
    print(f'written to {CACHE}')

judged.index = manual.index

### Agreement per field

Reported per field rather than pooled. The fields differ by an order of
magnitude in how often they occur, so a single figure would be decided by the
common ones.

`alternative_response` is conditional: it is scored only where the answer is a
refusal or nothing was delivered, and is a forced No everywhere else. Computing
its agreement over every row would credit both sides for a foregone answer on
three quarters of the corpus, so it is restricted to the rows where it varies.

In [ ]:
import numpy as np


# Define function to correct agreement for the share that would happen by chance
# given how often each value occurs. Returns nothing where one value fills the
# column: that is an undefined coefficient rather than a low one.
def kappa(left, right):
    pairs = [(a, b) for a, b in zip(left, right)
             if str(a).strip() and str(b).strip()]
    if not pairs:
        return None, 0, None
    agreed = sum(a == b for a, b in pairs) / len(pairs)
    values = {value for pair in pairs for value in pair}
    expected = sum((sum(a == v for a, _ in pairs) / len(pairs))
                   * (sum(b == v for _, b in pairs) / len(pairs))
                   for v in values)
    if expected >= 1:
        return None, len(pairs), agreed
    return round((agreed - expected) / (1 - expected), 3), len(pairs), agreed


# Rows where the annotator was unsure are excluded: a coefficient is a claim
# about labels the annotator stood behind.
sure = manual.get('uncertain', pd.Series('', index=manual.index)) != 'unsure'

# Alternative varies only where something was withheld. Everywhere else it is a
# forced No on both sides.
scorable = ((manual['answer'] == 'Refusal')
            | (manual['delivery_response'] == 'No'))

# Define function to give the confusion counts and the measures that survive an
# unbalanced class. Kappa alone is not enough here: several fields are positive
# on under a twentieth of replies, and raw accuracy on those is high whatever the
# classifier does. Precision says how much of what it flagged was real, recall
# how much of what was there it found, and MCC is the one summary that does not
# flatter a classifier for getting the majority class right.
def measures(human, judge, positive):
    tp = int(((human == positive) & (judge == positive)).sum())
    fp = int(((human != positive) & (judge == positive)).sum())
    fn = int(((human == positive) & (judge != positive)).sum())
    tn = int(((human != positive) & (judge != positive)).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = (2 * precision * recall / (precision + recall)
          if precision and recall else None)
    denominator = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / denominator) if denominator else None
    return tp, fp, fn, tn, precision, recall, f1, mcc


report = []
for column in LABEL_COLUMNS:
    mask = sure & (scorable if column == 'alternative_response' else True)
    left, right = manual.loc[mask, column], judged.loc[mask, column]
    score, count, raw = kappa(left, right)
    positive = 'Refusal' if column == 'answer' else 'Yes'
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    report.append({
        'field': column,
        'kappa': score,
        'mcc': None if mcc is None else round(mcc, 3),
        'precision': None if precision is None else round(precision, 3),
        'recall': None if recall is None else round(recall, 3),
        'f1': None if f1 is None else round(f1, 3),
        'raw': None if raw is None else round(raw, 3),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'human': round(100 * (left == positive).mean(), 1),
        'judge': round(100 * (right == positive).mean(), 1),
        'n': count})

agreement = pd.DataFrame(report).set_index('field')
agreement.to_csv(JUDGE_DIR / 'agreement.csv')

def shown(value, places=3):
    return '-' if value is None or pd.isna(value) else f'{value:.{places}f}'


print(f'  {"field":24}{"kappa":>7}{"MCC":>7}{"prec":>6}{"rec":>6}{"F1":>6}'
      f'{"tp":>5}{"fp":>4}{"fn":>4}{"n":>6}')
for field, row in agreement.iterrows():
    # A field where one value fills the column has no coefficient. Printed as
    # flat rather than as a number, so it is never read as a low kappa.
    k = 'flat' if row['kappa'] is None or pd.isna(row['kappa']) \
        else f'{row["kappa"]:.3f}'
    note = '  conditional' if field == 'alternative_response' else ''
    print(f'  {field:24}{k:>7}{shown(row["mcc"]):>7}{shown(row["precision"],2):>6}'
          f'{shown(row["recall"],2):>6}{shown(row["f1"],2):>6}'
          f'{int(row["tp"]):>5}{int(row["fp"]):>4}{int(row["fn"]):>4}'
          f'{int(row["n"]):>6}{note}')
print(f'\nwritten to {JUDGE_DIR / "agreement.csv"}')

### Every row, side by side

`rows.csv` carries both sets of labels on every row, so a disagreement can be
read against the reply that produced it. `disagreements.csv` is the subset where
they differ, which is the file to read before changing the policy.

In [ ]:
both = manual[['model', 'prompt_id', 'replicate', 'stratum', 'prompt',
                'response']].copy()
for column in LABEL_COLUMNS:
    both[f'{column}_human'] = manual[column]
    both[f'{column}_judge'] = judged[column]
both['differ'] = sum((manual[c] != judged[c]).astype(int) for c in LABEL_COLUMNS)
both.to_csv(JUDGE_DIR / 'rows.csv', index=False)

apart = both[both['differ'] > 0].sort_values('differ', ascending=False)
apart.to_csv(JUDGE_DIR / 'disagreements.csv', index=False)

print(f'{len(both)} rows written to rows.csv')
print(f'{len(apart)} rows differ on at least one field '
      f'({len(apart) / len(both):.0%}), written to disagreements.csv')
print(f'{int(both["differ"].sum())} cells of {len(both) * len(LABEL_COLUMNS):,} '
      f'({both["differ"].sum() / (len(both) * len(LABEL_COLUMNS)):.1%})')
print()
print('the fields they differ on most:')
for column in LABEL_COLUMNS:
    n = int((manual[column] != judged[column]).sum())
    if n:
        print(f'  {column:24}{n:>4}')

### Where they differ, in the text

The first few of each, so a disagreement can be read rather than counted.

In [ ]:
SHOW = 3

for column in LABEL_COLUMNS:
    differ = both[both[f'{column}_human'] != both[f'{column}_judge']]
    if differ.empty:
        continue
    print('=' * 76)
    print(f'{column}: {len(differ)} of {len(both)}')
    print('=' * 76)
    for row in differ.head(SHOW).itertuples():
        human = getattr(row, f'{column}_human')
        judge = getattr(row, f'{column}_judge')
        print(f'\n  {row.model}  {row.prompt_id}  ({row.stratum})')
        print(f'    you {human}, the classifier {judge}')
        print(f'    {str(row.response)[:200]}'.replace(chr(10), ' '))
    print()

## Then

Read `disagreements.csv` before changing anything. A field the classifier and
the annotator disagree on in one direction is a definition to sharpen; a field
they disagree on in both directions is noise, and sharpening will not help it.

The policy is frozen once the full pass starts:

    python scripts/evaluate.py --backend ollama --workers 16

---

## Standalone: strip the annotator's columns

`uncertain`, `comment` and `note` are working notes rather than labels. They
belong in the sheets while labelling and not in a released file: `note` is the
pre-annotator's flag, `comment` may carry anything the annotator typed, and
`uncertain` is a property of the labelling session rather than of the reply.

This cell writes a cleaned copy beside each file rather than overwriting it, so
the working sheets keep their notes. Run it on its own; it needs nothing above
except the imports.

In [ ]:
ASIDE = ['uncertain', 'comment', 'note']
SOURCES = [settings.RESULTS_DIR / 'annotation manual',
           settings.RESULTS_DIR / 'annotation']
CLEAN_DIR = settings.RESULTS_DIR / 'annotation released'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

cleaned = 0
for folder in SOURCES:
    if not folder.exists():
        print(f'  {folder.name}: not present, skipped')
        continue
    target = CLEAN_DIR / folder.name.replace('annotation ', '').replace(
        'annotation', 'pre')
    target.mkdir(parents=True, exist_ok=True)
    for path in sorted(folder.glob('*.csv')):
        sheet = pd.read_csv(path, dtype=str, keep_default_na=False)
        dropped = [c for c in ASIDE if c in sheet.columns]
        if not dropped and 'answer' not in sheet.columns:
            continue                      # agreement tables, not label sheets
        sheet = sheet.drop(columns=dropped)
        sheet.to_csv(target / path.name, index=False)
        cleaned += 1
        print(f'  {folder.name}/{path.name:<46} dropped {dropped or "nothing"}')

print(f'\n{cleaned} files written to {CLEAN_DIR}')
print('The originals keep their notes; nothing above was overwritten.')